In [ ]:
ion.


# === SETUP: load the provided data files (regenerate them if missing) ===
import os
import numpy as np
import pandas as pd

def build_datasets(csv_path='ecommerce_customers.csv',
                   xlsx_path='transactions.xlsx', seed=42, verbose=False):
    """Generate a realistic e-commerce customer + transactions dataset.

    Baked-in realism for EDA / feature engineering practice:
      - right-skewed monetary columns (total_spend) with a few 'whale' outliers
      - class imbalance in is_churned (~20-30%)
      - real signal: churn depends on recency, order count and support tickets
      - missing values in age / gender / city
      - a high-cardinality 'city' column (long tail of rare cities)
      - customers with zero orders (no last_order_date) -> dormant
    The Excel file is order-level and is CONSISTENT with the customer table
    (num_orders / total_spend / last_order_date are derived from it).
    """
    rng = np.random.default_rng(seed)
    N = 2500
    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2024-06-30')
    horizon = (end - start).days

    cust = np.array([f'CUST{i+1:05d}' for i in range(N)])
    signup_off = rng.integers(0, horizon - 60, N)
    signup = start + pd.to_timedelta(signup_off, unit='D')

    # order counts: overdispersed (gamma-poisson), some customers have zero
     lam = rng.gamma(2.0, 1.6, N)
    num_orders = rng.poisson(lam)

    # ---- order-level transactions (vectorised) ----
    counts = num_orders
    tot = int(counts.sum())
    cust_rep = np.repeat(cust, counts)
    signup_rep = np.repeat(signup_off, counts)
    span = np.maximum(horizon - signup_off, 1)
    span_rep = np.repeat(span, counts)
    off = (rng.random(tot) * span_rep).astype(int)
    tx_off = signup_rep + off
    tx_date = start + pd.to_timedelta(tx_off, unit='D')
    amount = rng.lognormal(3.2, 0.8, tot).round(2)        # right-skewed (~tens of currency)
    category = rng.choice(['Electronics', 'Fashion', 'Grocery', 'Home', 'Books'],
                          tot, p=[.20, .30, .25, .15, .10])
    tx = pd.DataFrame({'customer_id': cust_rep, 'order_date': tx_date,
                       'amount': amount, 'category': category}).sort_values(
        ['customer_id', 'order_date']).reset_index(drop=True)

    # ---- aggregate transactions -> customer level ----
    agg = tx.groupby('customer_id').agg(
        total_spend=('amount', 'sum'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    ).reset_index()
     df = pd.DataFrame({'customer_id': cust, 'signup_date': signup,
                       'num_orders': num_orders})
    df = df.merge(agg, on='customer_id', how='left')
    df['total_spend'] = df['total_spend'].fillna(0).round(2)

    # ---- demographics & account attributes ----
    df['age'] = np.clip(rng.normal(38, 12, N), 18, 82).round().astype(int)
    df['gender'] = rng.choice(['M', 'F', 'Other'], N, p=[.48, .48, .04])

    majors = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    rare = ['Jaipur', 'Surat', 'Indore', 'Bhopal', 'Patna', 'Nagpur',
            'Kochi', 'Coimbatore', 'Visakhapatnam', 'Lucknow']
    pool = majors + rare
    w = np.array([.17, .15, .14, .12, .10, .08, .06] + [.013] * 10)
    w = w / w.sum()
    df['city'] = rng.choice(pool, N, p=w)

    df['plan'] = rng.choice(['Basic', 'Standard', 'Premium'], N, p=[.50, .35, .15])
    df['device'] = rng.choice(['Mobile', 'Desktop', 'Tablet'], N, p=[.60, .32, .08])
    df['payment_method'] = rng.choice(['Card', 'UPI', 'Wallet', 'NetBanking'],
                                      N, p=[.40, .35, .15, .10])
    df['support_tickets'] = rng.poisson(0.6, N)
    df['email_opt_in'] = rng.choice([0, 1], N, p=[.35, .65])

    # ---- churn target with real signal (recency / orders / tickets) ----
    last = pd.to_datetime(df['last_order'])
    days_since = (end - last).dt.days
    days_since_filled = days_since.fillna(horizon).to_numpy()
    z = (-2.75
         + 0.0019 * days_since_filled
         + 0.30 * df['support_tickets'].to_numpy()
         - 0.05 * df['num_orders'].to_numpy()
         + 0.70 * (df['num_orders'].to_numpy() == 0))
    p = 1 / (1 + np.exp(-z))
    df['is_churned'] = (rng.random(N) < p).astype(int)
     # ---- format dates as ISO strings (NaT -> ) ----
    df = df.rename(columns={'first_order': 'first_order_date',
                            'last_order': 'last_order_date'})
    for c in ['signup_date', 'first_order_date', 'last_order_date']:
        df[c] = pd.to_datetime(df[c]).dt.date.astype('string')

    df = df[['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'age', 'gender', 'city', 'plan', 'device', 'payment_method',
             'num_orders', 'total_spend', 'support_tickets', 'email_opt_in',
             'is_churned']]

    # ---- inject missing values AFTER computing the target ----
    def punch(col, frac):
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    punch('age', 0.07)
    punch('gender', 0.04)
    punch('city', 0.02)

    # ---- write files ----
    df.to_csv(csv_path, index=False)
    tx_out = tx.copy()
    tx_out['order_date'] = pd.to_datetime(tx_out['order_date']).dt.date.astype('string')
    tx_out.to_excel(xlsx_path, index=False)

    if verbose:
        print('customers:', df.shape, '| transactions:', tx_out.shape)
        print('churn rate:', round(df["is_churned"].mean(), 3))
        print('total_spend skew:', round(df["total_spend"].skew(), 2))
        print('missing age:', int(df["age"].isna().sum()),
              '| missing city:', int(df["city"].isna().sum()))
        print('zero-order customers:', int((df["num_orders"] == 0).sum()))
        print('distinct cities:', df["city"].nunique())
    return df, tx_out
if not (os.path.exists('ecommerce_customers.csv') and os.path.exists('transactions.xlsx')):
    build_datasets()
    print('Generated dataset files.')
else:
    print('Found the provided dataset files.')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SNAPSHOT = pd.Timestamp('2024-06-30')   # the day the data was pulled

cust = pd.read_csv('ecommerce_customers.csv',
                   parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
tx = pd.read_excel('transactions.xlsx', parse_dates=['order_date'])
print('customers:', cust.shape, '| transactions:', tx.shape)
tx.head()



In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SNAPSHOT = pd.Timestamp('2024-06-30')   # the day the data was pulled

cust = pd.read_csv('ecommerce_customers.csv',
                   parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
tx = pd.read_excel('transactions.xlsx', parse_dates=['order_date'])
print('customers:', cust.shape, '| transactions:', tx.shape)
tx.head()


In [ ]:

# -----------------------------------------------------------
# 🔹 1A. ROLL UP ORDER-LEVEL ROWS TO ONE ROW PER CUSTOMER
# -----------------------------------------------------------
rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),                 # F: how many orders
    monetary=('amount', 'sum'),                   # M: total spent
    avg_order_value=('amount', 'mean'),           # typical basket
    last_order=('order_date', 'max'),
    n_categories=('category', 'nunique'),         # breadth of interest
).reset_index()
# R: recency = days since last order, measured from the snapshot
rfm['recency_days'] = (SNAPSHOT - rfm['last_order']).dt.days
rfm[['customer_id', 'frequency', 'monetary', 'avg_order_value',
     'recency_days', 'n_categories']].head()

In [ ]:

# -----------------------------------------------------------
# 🔹 1B. MERGE THE NEW FEATURES BACK ONTO THE CUSTOMER TABLE
# -----------------------------------------------------------
df = cust.merge(rfm.drop(columns='last_order'), on='customer_id', how='left')
# customers with no transactions -> fill sensible defaults
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
df['n_categories'] = df['n_categories'].fillna(0)
# never ordered -> recency is 'infinitely' stale: cap at customer tenure
df['recency_days'] = df['recency_days'].fillna(9999)
print('Merged shape:', df.shape)
df[['customer_id', 'frequency', 'monetary', 'recency_days', 'is_churned']].head()

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. first & last order per customer -> active span in days
order_spans = tx.groupby('customer_id')['order_date'].agg(first_order='min', last_order='max').reset_index()
order_spans['active_span_days'] = (order_spans['last_order'] - order_spans['first_order']).dt.days

# 2. avg_days_between_orders = span / (frequency - 1), guard freq<=1
# We temporarily bring in frequency from our rfm table to compute this safely
order_spans = order_spans.merge(rfm[['customer_id', 'frequency']], on='customer_id', how='left')

order_spans['avg_days_between_orders'] = np.where(
    order_spans['frequency'] > 1,
    order_spans['active_span_days'] / (order_spans['frequency'] - 1),
    np.nan  # Use NaN (or 0 / 9999 depending on your preference) to guard against zero-division
)

# 3. merge onto df and show head
df = df.merge(order_spans[['customer_id', 'active_span_days', 'avg_days_between_orders']], on='customer_id', how='left')

# Optional: fill missing span values for dormant customers who never ordered
df['active_span_days'] = df['active_span_days'].fillna(0)

# Display result
print('Final merged shape:', df.shape)
df[['customer_id', 'frequency', 'monetary', 'active_span_days', 'avg_days_between_orders', 'is_churned']].head()


In [ ]:
# -----------------------------------------------------------
# 🔹 2A. LOG-TRANSFORM A RIGHT-SKEWED COLUMN
# -----------------------------------------------------------
# EDA showed 'monetary' (total spend) is right-skewed. log1p compresses
# the long tail so big spenders don't dominate the model.
df['monetary_log'] = np.log1p(df['monetary'])   # log1p handles 0 safely
print('skew before:', round(df['monetary'].skew(), 2))
print('skew after :', round(df['monetary_log'].skew(), 2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(df['monetary'], bins=40, color='#3AAFA9'); ax[0].set_title('monetary (raw)')
ax[1].hist(df['monetary_log'], bins=40, color='#2D6A9F'); ax[1].set_title('monetary (log1p)')
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. skew of avg_order_value (note: NaN for zero-order customers filled with 0)
df['avg_order_value'] = df['avg_order_value'].fillna(0)
skew_before = df['avg_order_value'].skew()

# 2. log-transform
df['avg_order_value_log'] = np.log1p(df['avg_order_value'])

# 3. compare skew before/after
skew_after = df['avg_order_value_log'].skew()

print('skew before:', round(skew_before, 2))
print('skew after :', round(skew_after, 2))

# Optional: Visualize the transformation
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(df['avg_order_value'], bins=40, color='#E27D60')
ax[0].set_title('avg_order_value (raw)')
ax[1].hist(df['avg_order_value_log'], bins=40, color='#85DCB0')
ax[1].set_title('avg_order_value (log1p)')
plt.tight_layout()
plt.show()


In [ ]:

# -----------------------------------------------------------
# 🔹 3A. BUCKET AGE INTO LIFE-STAGE GROUPS
# -----------------------------------------------------------
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 40, 60, 200],
                         labels=['<25', '25-39', '40-59', '60+'])
print(df['age_group'].value_counts(dropna=False))


In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. bucket recency_days  (tip: use bins=[-1, 30, 90, 100000])
# Labels describe: Active within month, Recent (1-3 months), Stale/Dormant (>3 months)
df['recency_group'] = pd.cut(
    df['recency_days'],
    bins=[-1, 30, 90, 100000],
    labels=['0-30 days', '31-90 days', '91+ days']
)

# 2. counts per bucket
print("--- Counts Per Recency Bucket ---")
print(df['recency_group'].value_counts(dropna=False))
print("\n")

# 3. pd.crosstab(bucket, df['is_churned'], normalize='index')
# This gives the row-normalised proportions (i.e., churn rate per bucket)
churn_cross_tab = pd.crosstab(df['recency_group'], df['is_churned'], normalize='index')

print("--- Churn Proportions by Recency Group ---")
print(churn_cross_tab)


In [ ]:
# -----------------------------------------------------------
# 🔹 4A. RATIOS THAT EXPOSE BEHAVIOUR  (mind divide-by-zero!)
# -----------------------------------------------------------
# tickets per order: a dissatisfaction signal, normalised by activity
df['tickets_per_order'] = df['support_tickets'] / df['frequency'].replace(0, np.nan)
df['tickets_per_order'] = df['tickets_per_order'].fillna(0)   # 0 orders -> 0

# spend per category: concentration of spend
df['spend_per_category'] = df['monetary'] / df['n_categories'].replace(0, np.nan)
df['spend_per_category'] = df['spend_per_category'].fillna(0)
print(df[['support_tickets', 'frequency', 'tickets_per_order',
          'monetary', 'n_categories', 'spend_per_category']].head())

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. tenure in months, then orders_per_month (guard divide-by-zero)
tenure_months = (SNAPSHOT - df['signup_date']).dt.days / 30.0

# Using .replace(0, np.nan) on tenure ensures we don't divide by zero if someone signed up today
df['orders_per_month'] = df['frequency'] / tenure_months.replace(0, np.nan)

# 2. replace inf/NaN with 0 (hint: .replace([np.inf,-np.inf], np.nan).fillna(0))
df['orders_per_month'] = df['orders_per_month'].replace([np.inf, -np.inf], np.nan).fillna(0)

# 3. top 5 by frequency
print("--- Top 5 Customers by Order Frequency ---")
top_5_frequent = df.nlargest(5, 'frequency')
print(top_5_frequent[['customer_id', 'signup_date', 'frequency', 'orders_per_month', 'is_churned']])


In [ ]:

# -----------------------------------------------------------
# 🔹 5A. UNLOCK THE DATE COLUMNS
# -----------------------------------------------------------
df['tenure_days'] = (SNAPSHOT - df['signup_date']).dt.days   # how long a customer
df['signup_month'] = df['signup_date'].dt.month
df['signup_dow'] = df['signup_date'].dt.dayofweek            # 0=Mon
df['signup_is_weekend'] = (df['signup_dow'] >= 5).astype(int)
# cyclical encoding so Dec(12) and Jan(1) are 'close', not far apart
df['month_sin'] = np.sin(2 * np.pi * df['signup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['signup_month'] / 12)
df[['signup_date', 'tenure_days', 'signup_month',
    'signup_dow', 'signup_is_weekend', 'month_sin', 'month_cos']].head()


In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. days from first_order_date to SNAPSHOT
# Since df['first_order_date'] was cast to a string/object type, we ensure it is datetime first
days_since_first_order = (SNAPSHOT - pd.to_datetime(df['first_order_date'])).dt.days

# 2. fill NaT-derived NaN with 0
# Dormant customers who never ordered will output NaN from the date subtraction; we map them to 0
df['days_since_first_order'] = days_since_first_order.fillna(0).astype(int)

# 3. show a few rows
print("--- Days Since First Order Feature ---")
print(df[['customer_id', 'signup_date', 'first_order_date', 'days_since_first_order', 'frequency']].head(10))


In [ ]:
# -----------------------------------------------------------
# 🔹 6A. ONE-HOT THE LOW-CARDINALITY COLUMNS
# -----------------------------------------------------------
df = pd.get_dummies(df, columns=['plan', 'device', 'payment_method'],
                    drop_first=False, dtype=int)
print('columns after one-hot:', df.shape[1])
print([c for c in df.columns if c.startswith(('plan_', 'device_'))])

In [ ]:
# -----------------------------------------------------------
# 🔹 6B. FREQUENCY-ENCODE THE HIGH-CARDINALITY 'city'
# -----------------------------------------------------------
# one-hot on 17 cities would add 17 sparse columns; frequency encoding
# keeps it to ONE numeric column = how common each city is.
freq = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq).fillna(0)
print(df[['city', 'city_freq']].drop_duplicates().head(8))

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. fill missing gender with 'Unknown'
df['gender'] = df['gender'].fillna('Unknown')

# 2. one-hot encode gender
# We pass dtype=int to convert the resulting True/False booleans into numeric 1/0 flags
df = pd.get_dummies(df, columns=['gender'], drop_first=False, dtype=int)

# 3. show the new columns
print("--- New One-Hot Encoded Gender Columns ---")
gender_cols = [c for c in df.columns if c.startswith('gender_')]
print(gender_cols)
print("\nSample Rows:")
print(df[gender_cols].head())


In [ ]:

# -----------------------------------------------------------
# 🔹 7A. BUCKET RARE CITIES INTO 'Other'
# -----------------------------------------------------------
top_cities = df['city'].value_counts().head(7).index   # keep the 7 biggest
df['city_grouped'] = df['city'].where(df['city'].isin(top_cities), 'Other')
print('before:', df['city'].nunique(), 'cities')
print('after :', df['city_grouped'].nunique(), 'categories')
print(df['city_grouped'].value_counts())


In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. one-hot city_grouped
df = pd.get_dummies(df, columns=['city_grouped'], drop_first=False, dtype=int)

# 2. number of new city_grouped_* columns
new_city_cols = [c for c in df.columns if c.startswith('city_grouped_')]
print(f"Number of new city_grouped columns: {len(new_city_cols)}")
print("New columns:", new_city_cols)
print("\nSample Rows:")
print(df[new_city_cols].head())

# 3. Why group first:
# Grouping rare categories before one-hot encoding prevents the creation of
# a high number of sparse, highly uninformative binary features (the long tail).
# This limits variance, avoids the 'curse of dimensionality', protects the model
# from overfitting on tiny samples (e.g., a city with only 3 customers), and
# reduces the overall computation/memory footprint during training.


In [ ]:
# -----------------------------------------------------------
# 🔹 8A. ENCODE BUSINESS KNOWLEDGE AS FEATURES
# -----------------------------------------------------------
# a 'dormant' flag: no orders, or very stale
df['is_dormant'] = ((df['frequency'] == 0) | (df['recency_days'] > 180)).astype(int)
# a simple RFM score: rank each of R, F, M into 1-4 and combine
df['R_score'] = pd.qcut(-df['recency_days'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
df['F_score'] = pd.qcut(df['frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
df['M_score'] = pd.qcut(df['monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
df['rfm_score'] = df['R_score'] + df['F_score'] + df['M_score']   # 3..12
print(df[['recency_days', 'frequency', 'monetary', 'rfm_score', 'is_dormant']].head())
print('\nMean churn by RFM score:')
print(df.groupby('rfm_score')['is_churned'].mean().round(2))

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. high_value flag (top 20% of monetary)
# We use pd.qcut to group users into quintiles, marking the top 20% (quintile 5) as high value
df['high_value'] = (pd.qcut(df['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int) == 5).astype(int)

# 2. churn rate by high_value
print("--- Churn Rate Comparison (High Value vs Others) ---")
print(df.groupby('high_value')['is_churned'].mean().round(3))
print("\n")

# 3. Another domain feature I'd add:
# df['unresolved_complaint_risk'] = (df['support_tickets'] > 0) & (df['is_dormant'] == 1)
#
# Domain Context: I would flag 'at-risk' customer friction by intersecting support activity
# with transaction drop-off. A customer with high support tickets who hasn't bought anything
# recently is highly likely to be a "rage churn" who left due to bad service, which is a
# completely different psychological profile than a customer who simply forgot about the platform.


In [ ]:

# -----------------------------------------------------------
# 🔹 9A. BUILD X (features) AND y (target)
# -----------------------------------------------------------
drop_cols = ['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'city', 'gender', 'age_group', 'city_grouped', 'is_churned']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include='number')   # keep numeric features only
y = df['is_churned']
print('Feature matrix:', X.shape)
print('All numeric? ', X.select_dtypes(exclude='number').empty)
print('Any missing? ', int(X.isna().sum().sum()))
print('\nEngineered features include:')
print([c for c in X.columns if c in ('monetary_log','recency_days','rfm_score',
       'tickets_per_order','tenure_days','city_freq','is_dormant')])

In [ ]:
# -----------------------------------------------------------
# 🔹 YOUR COMPLETED CODE HERE
# -----------------------------------------------------------

# 1. fill remaining missing values with the median
# This targets columns like 'age' or 'avg_days_between_orders' which may still contain NaNs
X = X.fillna(X.median())

# 2. correlation of each feature with y, sorted
print("--- Feature Correlation with Churn (Target) ---")
correlations = X.assign(churn=y).corr()['churn'].drop('churn').sort_values()
print(correlations)
print("\n")

# 3. A feature that would be leakage:
# 'days_since_last_transaction' calculated using post-snapshot activity, or internal database flags
# like 'account_deletion_timestamp'.
#
# Domain Context: Including a feature that relies on information from the *future* (after the cutoff point)
# or directly mirrors the target variable creates an artificial perfect score during training. For instance,
# if 'is_churned' was calculated partly based on a customer having 0 orders in the last 180 days, adding
# a feature like 'total_orders_in_last_30_days' that spans into that same evaluation window would leak
# the target and cause the model to fail in production.
